In [11]:
import pandas as pd
import numpy as np
from scipy import stats

# -------------------------------------------------------
# STEP 1: Business Hypothesis
# -------------------------------------------------------
# Business question: "Do discounted orders lead to a higher average
#                      order value than non-discounted orders?"
#
# H0 (Null Hypothesis): There is NO significant difference in average
#                        net order value between discounted and
#                        non-discounted orders.
# H1 (Alternative Hypothesis): Discounted orders have a significantly
#                               different average net order value than
#                               non-discounted orders.

print("BUSINESS HYPOTHESIS")
print("-" * 50)
print("H0: No difference in avg order value (discount vs no discount)")
print("H1: Significant difference exists\n")

# -------------------------------------------------------
# STEP 2: Create dataset (same structure as Project 4)
# -------------------------------------------------------
np.random.seed(42)

data = {
    "transaction_id": range(1, 101),
    "quantity": np.random.randint(1, 6, 100),
    "unit_price": np.random.choice([500, 800, 1200, 25000, 45000], 100),
    "discount_pct": np.random.choice([0, 5, 10, 15, 20], 100),
    "region": np.random.choice(["North", "South", "East", "West"], 100),
}

df = pd.DataFrame(data)
df["net_amount"] = (df["quantity"] * df["unit_price"]) * (1 - df["discount_pct"] / 100)
df["had_discount"] = df["discount_pct"] > 0

# -------------------------------------------------------
# STEP 3: Split into two groups
# -------------------------------------------------------
discounted = df[df["had_discount"]]["net_amount"]
not_discounted = df[~df["had_discount"]]["net_amount"]

print(f"Discounted orders: n={len(discounted)}, mean={discounted.mean():.2f}")
print(f"Non-discounted orders: n={len(not_discounted)}, mean={not_discounted.mean():.2f}\n")

# -------------------------------------------------------
# STEP 4: Validate assumptions before testing
# -------------------------------------------------------

# Assumption check 1: Normality (Shapiro-Wilk test)
shapiro_disc = stats.shapiro(discounted)
shapiro_nodisc = stats.shapiro(not_discounted)
print("Normality Check (Shapiro-Wilk):")
print(f"  Discounted group p-value: {shapiro_disc.pvalue:.4f}")
print(f"  Non-discounted group p-value: {shapiro_nodisc.pvalue:.4f}")
print("  (p < 0.05 means data is NOT normally distributed)\n")

# Assumption check 2: Equal variances (Levene's test)
levene_test = stats.levene(discounted, not_discounted)
print("Equal Variance Check (Levene's Test):")
print(f"  p-value: {levene_test.pvalue:.4f}")
print("  (p < 0.05 means variances are significantly different)\n")

# -------------------------------------------------------
# STEP 5: Apply the appropriate statistical test
# -------------------------------------------------------
# Since real-world business data is often non-normal / skewed,
# we use the Mann-Whitney U test (non-parametric alternative to t-test).
# If data were normal with equal variance, an independent t-test would be used instead.

u_stat, p_value = stats.mannwhitneyu(discounted, not_discounted, alternative="two-sided")

print("STATISTICAL TEST: Mann-Whitney U Test")
print("-" * 50)
print(f"U-statistic: {u_stat:.2f}")
print(f"p-value: {p_value:.4f}\n")

# For comparison, also run a standard independent t-test
t_stat, t_pvalue = stats.ttest_ind(discounted, not_discounted, equal_var=False)
print("For Reference: Independent t-test (Welch's)")
print(f"t-statistic: {t_stat:.2f}, p-value: {t_pvalue:.4f}\n")

# -------------------------------------------------------
# STEP 6: Interpret results correctly
# -------------------------------------------------------
alpha = 0.05
print("INTERPRETATION")
print("-" * 50)
if p_value < alpha:
    print(f"p-value ({p_value:.4f}) < alpha (0.05) -> Reject H0.")
    print("There IS a statistically significant difference in order value")
    print("between discounted and non-discounted orders.")
else:
    print(f"p-value ({p_value:.4f}) >= alpha (0.05) -> Fail to reject H0.")
    print("There is NO statistically significant difference in order value")
    print("between discounted and non-discounted orders.")

print("\nBUSINESS TRANSLATION")
print("-" * 50)
print("This tells the business whether running discount campaigns actually")
print("changes the typical order size, or whether discounts are simply")
print("reducing margin without changing customer purchasing behavior.")

BUSINESS HYPOTHESIS
--------------------------------------------------
H0: No difference in avg order value (discount vs no discount)
H1: Significant difference exists

Discounted orders: n=74, mean=34767.03
Non-discounted orders: n=26, mean=36915.38

Normality Check (Shapiro-Wilk):
  Discounted group p-value: 0.0000
  Non-discounted group p-value: 0.0000
  (p < 0.05 means data is NOT normally distributed)

Equal Variance Check (Levene's Test):
  p-value: 0.8860
  (p < 0.05 means variances are significantly different)

STATISTICAL TEST: Mann-Whitney U Test
--------------------------------------------------
U-statistic: 840.50
p-value: 0.3415

For Reference: Independent t-test (Welch's)
t-statistic: -0.19, p-value: 0.8506

INTERPRETATION
--------------------------------------------------
p-value (0.3415) >= alpha (0.05) -> Fail to reject H0.
There is NO statistically significant difference in order value
between discounted and non-discounted orders.

BUSINESS TRANSLATION
---------------